# M6.A4 — 결측·이상치 처리 규칙 확정 + 검증 분할 설계

> 산출 근거: `docs/plan/ai/phase_06_model.md` M6.A4 · 이관 목록: `02_features.ipynb` §8
> 규칙 SSOT: `AI/data_prep/preprocess.py` (본 노트북은 근거·검증 기록) · 작성: 2026-07-27

**목적** — `ml_pipeline.md` §6의 "별도 확정 예정" 항목(보간 방식·IQR 계수 k)과 walk-forward
fold 경계를 **train 구간 기준**으로 확정하고, 후보 규칙들을 데이터로 비교·기각한 근거를 남긴다.
확정 규칙은 `preprocess.py`에 코드로, spec 반영문은 §5 총괄표로 산출한다.

**원칙** — 규칙의 *형태*(방법·계수)는 여기서 고정하되, 경계값 fit은 학습 시 각 fold의 train
구간에서만 수행(누수 방지). 서빙 시점에 알 수 없는 값(미래 월 유동인구 등)은 보간에도 쓰지 않는다.

> 🔒 **공개 저장소 데이터 정책** — 실매장 매출의 절대 금액(원 단위 수치·그림·표)은 커밋하지 않는다. 본 노트북은 **출력 제거 상태로 추적**되며, 본문 서술의 금액은 비율·배수로 대체했다. 전체 수치·그림은 로컬 재실행으로 전량 재현된다 (실행법: `AI/README.md`).

## 판정 요약 (TL;DR)

1. **검증 분할 확정** — **월 단위 walk-forward 6 fold**(검증 월: 2025-09~12, 2026-03~04).
   검증 fold = 영업일 10일 이상인 캘린더 월, train = 그 이전 전체 영업일(112→240일 확장).
   휴업 월은 자동 배제(2026-01 영업 0일, 2026-02는 3일뿐) — "휴업 구간이 통째로 검증 fold가 되지 않게"(EDA §9) 충족.
2. **보간 확정** — 유동인구 전월 값은 **ffill(직전 가용 월) + staleness 플래그**(현 데이터 최대 1개월).
   선형 보간은 누락 월의 다음 달 값을 참조해 **누수라서 금지**. 기상 NA(강수 1일)는 캘린더 선형 보간.
   lag 워밍업 NaN은 트리=네이티브 유지 / 선형·통계 모델=core lag 완비 249행만.
3. **타깃 이상치 확정** — **log1p 스케일 IQR k=3.0, train-fit, winsorize(경계 캡)** → 현 데이터 **플래그 0건**(개입 없음).
   기각 근거: raw k=1.5는 상단 19건(3월 실수요) 오탐, log k=1.5는 하단 3건(실제 한산일) 오탐,
   P1/P99 캡은 15건 과잉 개입. "이상치 없음"이 데이터가 준 답 — 절차는 재학습 대비로 유지.
4. **floating hold 판정: 1군 제외 확정** — ffill 보간 후 ablation에서 **+1.2% 악화(6 fold 중 5개 악화)**.
   월 프록시 의심(02_features §6)과 합치. 재평가 트리거: 누락 월 원본 확보 또는 일별 유동인구(세담터) 전환 시.
5. **M6.A5 참조점** — keep 20열·확정 fold 기준 참조 성능 확보 — fold 간 최대 3.5배 편차, 2026-03(재개장+개강) fold가 최대 (절대액은 로컬 출력 참조). 모델 선정·튜닝은 M6.A5에서.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

_here = Path.cwd()
AI_DIR = next(p for p in [_here.parent, _here, _here / "AI"] if (p / "data_prep").exists())
sys.path.insert(0, str(AI_DIR / "data_prep"))
import preprocess as pp  # 규칙 SSOT

PAL = {"blue": "#2a78d6", "orange": "#eb6834", "gray": "#d9d8d4", "ink2": "#52514e"}
_installed = {f.name for f in fm.fontManager.ttflist}
plt.rcParams.update({
    "font.family": [f for f in ("AppleGothic", "Apple SD Gothic Neo", "NanumGothic") if f in _installed] or ["sans-serif"],
    "axes.unicode_minus": False, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 100,
})

feat = pd.read_parquet(AI_DIR / "data/processed/features_daily.parquet").set_index("date")
ob = feat[feat.is_open].copy()
y_raw = ob.total_amount
print(f"모델링 행 {len(ob)}일 | 규칙 상수: 검증 시작 {pp.FIRST_VAL_MONTH}, "
      f"검증 월 최소 영업일 {pp.VAL_MONTH_MIN_OPEN_DAYS}, 이상치 k={pp.OUTLIER_K} (log1p IQR)")

In [ ]:
# §1 검증 분할 — 월 단위 walk-forward
folds = pp.make_monthly_folds(ob.index)
fold_tbl = pd.DataFrame([{"검증 월": f["month"], "train 영업일": len(f["train"]),
                          "train 끝": f["train"].max().date(), "val 영업일": len(f["val"])}
                         for f in folds])
display(fold_tbl)
print("배제된 월: 2026-01(영업 0일 — 휴업), 2026-02(3일 < 최소 10일)")

fig, ax = plt.subplots(figsize=(11, 3.2), constrained_layout=True)
ax.axvspan(pd.Timestamp("2025-12-21"), pd.Timestamp("2026-02-26"), color=PAL["gray"], alpha=0.55, zorder=0)
for i, f in enumerate(folds):
    ax.barh(i, (f["train"].max() - f["train"].min()).days, left=f["train"].min(),
            height=0.55, color=PAL["blue"], alpha=0.8)
    ax.barh(i, (f["val"].max() - f["val"].min()).days + 1, left=f["val"].min(),
            height=0.55, color=PAL["orange"])
ax.set_yticks(range(len(folds)), [f["month"] for f in folds])
ax.set_title("월 단위 walk-forward — 파랑=train, 주황=검증 월, 회색=장기 휴업")
ax.margins(x=0.01); ax.grid(axis="y", visible=False)
plt.show()

### §1 관찰 — 분할 설계 근거

- 위치 기반 `TimeSeriesSplit` 대신 **월 경계**를 쓰는 이유: ① fold 성능이 "몇 월을 맞혔나"로
  읽혀 운영 보고와 일치 ② 휴업 월이 규칙(영업일<10)으로 자동 배제 ③ 개강월(9월)·재개장월(3월) 같은
  어려운 구간이 별도 fold로 분리돼 regime 적응력이 그대로 드러남.
- 최소 학습 이력 4개월(2025-04~08, 112 영업일) 확보 후 검증 시작 — lag·rolling 워밍업(최대 4주)을
  train 초입에 가둬 검증 구간 피처는 항상 완비.
- 최종 모델 확정 시 hold-out: **마지막 fold(2026-04)를 test로 봉인**하고 나머지 5개로 선택하는
  운영을 M6.A5에서 적용(이중 사용 방지).

In [ ]:
# §2 보간 규칙 — 기상(선형) · 유동인구(ffill+staleness) · lag 워밍업(이원화)
_, n_wx = pp.impute_weather(feat)
na_day = feat[feat[["temp_avg", "precip_mm"]].isna().any(axis=1)].index
prev_v, next_v = feat.loc["2025-08-03", "precip_mm"], feat.loc["2025-08-05", "precip_mm"]
print(f"기상 보간 대상 {n_wx}건: {[d.date() for d in na_day]} — 전일 {prev_v}mm·익일 {next_v}mm → 선형 보간값 {(prev_v+next_v)/2}mm")

imp = pp.impute_floating(ob)
show = imp[["floating_prev_m", "floating_prev_ffill", "floating_prev_stale"]].groupby(
    imp.index.strftime("%Y-%m")).first()
show.columns = ["원본 전월값", "ffill 보간", "staleness(개월)"]
display(show)

mask = pp.linear_model_mask(ob)
print(f"lag 워밍업: 트리 모델 = NaN 네이티브(256행 전부) / 선형·통계 모델 = core lag 완비 {int(mask.sum())}행")

### §2 관찰 — 보간 근거

- **유동인구에 선형 보간을 쓰지 않는 이유(누수)**: 누락 월 M의 선형 보간값은 M+1 실측을 참조하는데,
  서빙 시점(M 또는 M+1 초)엔 그 값이 아직 없다. ffill(직전 가용 월)은 어느 시점에도 재현 가능하고,
  누락 월이 전부 고립(연속 없음)이라 **staleness가 최대 1개월** — 정보 손실이 작다.
- 기상은 과거 실측 구간 보간이라 방향 제약이 없다 — 판매 기간 내 NA는 강수 1일(2025-08-04)뿐,
  전일·익일 모두 0mm라 보간값도 0mm(사실상 무영향).
- 워밍업 NaN을 "보간"하지 않는 이유: lag가 없다는 사실 자체가 정보(개업 직후)이고,
  트리 모델은 NaN 분기를 학습한다. 선형 모델 비교(M6.A5)에서만 249행으로 정렬.

In [ ]:
# §3 타깃 이상치 — 후보 5안 비교 (경계 fit = 최종 fold train ~2026-02-28, 적용 = 전체 256일)
tr_end = folds[-2]["train"].max()  # 2026-03 fold의 train 끝 = 2026-02-28
tr_raw = y_raw.loc[:tr_end]
rows = []
for label, s_tr, s_all in [("raw", tr_raw, y_raw), ("log1p", np.log1p(tr_raw), np.log1p(y_raw))]:
    for k in (1.5, 3.0):
        q1, q3 = s_tr.quantile([0.25, 0.75]); iqr = q3 - q1
        rows.append((f"{label} IQR k={k}", int((s_all > q3 + k * iqr).sum()), int((s_all < q1 - k * iqr).sum())))
p1, p99 = tr_raw.quantile([0.01, 0.99])
rows.append(("winsorize P1/P99", int((y_raw > p99).sum()), int((y_raw < p1).sum())))
cand = pd.DataFrame(rows, columns=["후보 규칙", "상단 플래그", "하단 플래그"])
cand["판정"] = ["기각 — 상단이 전부 실수요(3월 재개장 수준·EDA §2.4)", "기각 — raw 상단 3건도 실수요 스파이크",
              "기각 — 하단 3건이 실제 한산일(아래 표)", "✅ 확정 — 개입 0건",
              "기각 — 실수요 15건 과잉 캡"]
display(cand)

lo_flag = np.log1p(y_raw) < np.log1p(tr_raw).quantile(0.25) - 1.5 * (np.log1p(tr_raw).quantile(0.75) - np.log1p(tr_raw).quantile(0.25))
low3 = ob.loc[lo_flag, ["total_amount", "tx_count"]]
low3["실체"] = ["한산일(주문 1건, 실거래)", "한산일(주문 1건, 실거래)", "재개장 첫날 소프트 오픈(EDA §2.2)"]
print("log k=1.5가 잡는 하단 3일 — 오류가 아닌 실수요라 제거·캡 부적절:")
display(low3)

bounds = pp.outlier_bounds(tr_raw)
capped = pp.winsorize_target(y_raw, bounds)
print(f"확정 규칙 log1p IQR k={pp.OUTLIER_K} → 경계 {bounds[0]:,.0f} ~ {bounds[1]:,.0f}원, "
      f"캡 적용 {int((capped != y_raw).sum())}건")

### §3 관찰 — "이상치 없음"이 결론인 이유

- EDA §2.4에서 raw 스케일 상단 10건은 대부분 재개장 후 새 regime의 정상 수준으로 판정됐다.
  학습 타깃이 log1p인 이상, 이상치 판정도 같은 스케일이 정합적 — **log 스케일에선 상·하단 모두
  k=3.0 경계 안**이다(왜도 1.49가 log 후 거의 대칭이 된 결과).
- 순수 스파이크로 보였던 2025-09-11(주문 6건의 단체 추정 고액일)도 log 경계 안 → 개입 근거 없음.
- 규칙을 "없음"이 아니라 **절차(log IQR k=3.0 train-fit winsorize)** 로 남기는 이유: 운영 재학습에서
  POS 오류·이벤트성 진짜 이상값이 유입될 때 자동 방어선이 되고, 현 데이터에선 무개입이라 부작용도 없다.

In [ ]:
# §4 floating hold 재평가 — ffill 보간 후 ablation (02_features §7 keep 셋 기준)
import lightgbm as lgb

KEEP = ["is_holiday", "semester_week", "is_semester_first2w", "temp_avg", "temp_range",
        "lag1_sales", "lag1_tx", "lag_dow_sales", "roll7_mean", "roll4dow_mean", "roll7_atv",
        "is_post_renewal", "days_since_reopen"]

def matrix(df, extra=()):
    X = df[KEEP + list(extra)].copy()
    X = pd.concat([X, pd.get_dummies(df.index.dayofweek, prefix="dow").set_index(df.index)], axis=1)
    b = X.select_dtypes(bool).columns
    X[b] = X[b].astype(int)
    return X

def run_folds(X):
    y = np.log1p(ob.total_amount)
    maes = []
    for f in folds:
        m = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, num_leaves=15,
                              min_child_samples=10, subsample=0.9, colsample_bytree=0.9,
                              random_state=42, verbosity=-1)
        m.fit(X.loc[f["train"]], y.loc[f["train"]], eval_set=[(X.loc[f["val"]], y.loc[f["val"]])],
              callbacks=[lgb.early_stopping(50, verbose=False)])
        maes.append(np.mean(np.abs(np.expm1(m.predict(X.loc[f["val"]])) - ob.total_amount.loc[f["val"]])))
    return maes

mae_base = run_folds(matrix(ob))
mae_flo = run_folds(matrix(imp, ["floating_prev_ffill", "floating_prev_stale"]))
abl = pd.DataFrame({"keep 20열": mae_base, "+ floating(ffill+stale)": mae_flo},
                   index=[f["month"] for f in folds])
abl.loc["평균"] = abl.mean()
display(abl.round(0).astype(int).map(lambda v: f"{v:,}"))
print(f"floating 추가 효과: {(np.mean(mae_flo)/np.mean(mae_base)-1)*100:+.1f}% "
      f"(악화 fold {int(sum(f>b for b, f in zip(mae_base, mae_flo)))}/6)")

### §4 관찰 — floating 1군 제외 확정

- 보간을 제대로 해줘도 **+1.2% 악화, 6개 fold 중 5개에서 악화** — 02_features §6의 "월 더미 프록시"
  의심과 합쳐 **M6.A5 1군에서 제외**한다. permutation 상위였던 것은 월 식별 착시였다는 쪽으로 정리.
- 규칙(ffill+staleness)은 `preprocess.py`에 유지 — **재평가 트리거**: ① 누락 7개월 원본 확보(검수 대기)
  ② 일별 유동인구(세담터, Phase 8 운영 수집) 전환 시. 월간 해상도 자체가 학기 플래그와 정보 중복이라
  일별 전환 전엔 기대치 낮음.

## §5 확정 규칙 총괄 — `ml_pipeline.md` §6 반영문

| 항목 | 확정 규칙 | 근거 |
|---|---|---|
| 검증 분할 | 월 단위 walk-forward. 검증 fold = 영업일 ≥10일 월(2025-09부터), train = 이전 전체 영업일. 최종 fold는 test로 봉인 | §1 — 휴업 월 자동 배제, regime fold 분리 |
| 결측 — 유동인구 | 전월 값 ffill(직전 가용 월) + staleness 플래그. 선형 보간 금지(미래 월 참조 = 누수) | §2 — staleness ≤1개월 |
| 결측 — 기상 | 판매 기간 내 NA(강수 1일) 캘린더 선형 보간 | §2 — 과거 실측, 무영향 확인 |
| 결측 — lag 워밍업 | 트리 모델 NaN 네이티브 / 선형·통계 모델 core lag(lag1·roll7·lag_dow) 완비 249행 | §2 — "lag 없음"도 정보 |
| 타깃 이상치 | log1p IQR **k=3.0**, 각 fold train에 fit, 경계 밖 winsorize. 현 데이터 개입 0건 | §3 — raw·k1.5·P1/P99 전부 실수요 오탐으로 기각 |
| 피처 이상치 | 별도 처리 없음(기상 극값은 실측, 트리 모델 스케일 불변) | 02_features §5 |

구현: `AI/data_prep/preprocess.py` (상수 `FIRST_VAL_MONTH`·`VAL_MONTH_MIN_OPEN_DAYS`·`OUTLIER_K`).

## §6 판정·다음 단계

**M6.A4 종료 판정** — `ml_pipeline.md` §6의 "별도 확정 예정" 2건(보간 방식·IQR k)과 fold 경계까지
확정 + 후보 기각 근거 기록 + floating hold 해소(제외). **처리 규칙 문서화 산출 충족.**
spec 반영은 main 전용 문서 절차(docs 브랜치 PR)로 별도 수행.

### M6.A5(베이스라인 비교)로 넘어가는 것

- **입력**: keep 20열(02_features §7) — floating 제외 확정 반영. hold 잔여: is_exam(검수 대기)·month sin/cos·roll7_alcohol_share·days_gap_prev_open(각 ablation)
- **하네스**: 본 노트북 §1 fold 6개 + §3 winsorize 절차(무개입) + log1p 타깃, 지표 MAE 중심
  (+sMAPE 보조 — MAPE는 소액일 왜곡). **참조점: LGBM 스크리닝 fold별 MAE(§4 표 — 로컬 출력)**
- **비교군**(plan M6.A5): 이동평균·(S)ARIMA/Prophet·LightGBM (여력 시 LSTM·TimeXer), AutoGluon-TS 보조
- 마지막 fold(2026-04)는 test 봉인 — 모델 선택은 앞 5개 fold로

### 검수 연계 (변동 없음)

휴업 사유(학습 구간 3안 최종 확정용)·유동인구 원본(재평가 트리거)·학사일정 시험주간(is_exam hold)·
holidays 2025-10-10 — `AI/data/README.md` 검수 섹션.